# 05C Skill 与 MCP 一起进入 Agent 循环

我们已经分别验证了 MCP 工具连接和 Skill 加载。现在用第二节的模型接口，把二者接在同一个循环里。

先运行一次**教学回放**验证路由和真实本地工具；再读取随课**真实 DeepSeek 记录**核验模型选择。两种证据用途不同，不相互替代。

**贯穿案例**

把“解释配置入口和缺失文件分支”交给同一 Agent：它既可能激活本地仓库导读 Skill，也可能通过 MCP 读取源码。固定回放先验证连接与路由；另外读取已保存的真实模型运行，按工具轨迹和实际源码核对答案。回放的成功不代表真实模型已经答对。


## 学习路线与配套课件

实践 5C：联合运行并输出证据报告（`L05-P03`）。

建议先完成本节概念正课，再进入本实践小节。这个 Notebook 可用独立新内核从头运行。先预测，再执行代码、修改一个条件并解释结果。

对应课件稳定编号：L05.07-S01、L05.07-S02。页数调整不改变这些编号。

按“问题—代码—观察—练习—可复用结果”的顺序学习。先完成练习，再展开参考分析。回放、保存的真实记录和新的在线请求都会明确标注；在线请求默认关闭。

In [1]:
from pathlib import Path
import sys, json, asyncio, importlib.metadata, hashlib, os
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "00-资料库使用说明.md").exists())
CHAPTER = ROOT / "05 MCP 与 Skills 能力扩展"
sys.path.insert(0, str(CHAPTER / "代码"))
sys.path.insert(0, str(ROOT / "02 从大模型到 Agent 组成与最小循环/代码"))
print("MCP SDK:", importlib.metadata.version("mcp"))
print("课堂协议: 2026-07-28；本地 stdio；生成服务仍为 DeepSeek。")
from skill_mcp_agent import SKILL_TOOLS, SYSTEM, mcp_declarations, dispatch, run_skill_agent
from skill_lab import SkillLibrary
from mcp_connection import connect
from mcp import MCPError
from agent_lab import ReplayModel, tool_response, DeepSeekModel, ModelError, list_deepseek_models


MCP SDK: 2.1.1
课堂协议: 2026-07-28；本地 stdio；生成服务仍为 DeepSeek。


## 1. 两种工具来源，统一模型声明

`activate_skill`、`read_skill_resource` 是本地宿主工具。`mcp_list_files`、`mcp_read_file` 来自实际发现的 MCP 服务。本课加 `mcp_` 前缀，避免名字混淆，这不是 MCP 规范要求。

下面把 MCP 工具的输入 schema 适配成 DeepSeek 接口的工具声明。服务器返回内容后，再装入模型所需的 `tool` 消息。模型的调用 ID 与协议消息 ID 分别维护。


In [2]:
def mcp_declarations(tools):
    """把发现的 MCP inputSchema 适配为模型的工具声明；不是 MCP 协议本身。"""
    return [{'type': 'function', 'function': {'name': 'mcp_' + t.name,
             'description': t.description or '',
             'parameters': t.model_dump(by_alias=True)['inputSchema']}} for t in tools]


In [3]:
async with connect() as client:
    tools = (await client.list_tools()).tools
    model_tools = SKILL_TOOLS + mcp_declarations(tools)
print("模型可选:", [x["function"]["name"] for x in model_tools])
print(json.dumps(model_tools[-1], ensure_ascii=False, indent=2))


模型可选: ['activate_skill', 'read_skill_resource', 'mcp_list_files', 'mcp_read_file']
{
  "type": "function",
  "function": {
    "name": "mcp_read_file",
    "description": "读取 list_files 返回的一个文本文件，返回原文和行号。",
    "parameters": {
      "type": "object",
      "properties": {
        "path": {
          "title": "Path",
          "type": "string"
        }
      },
      "required": [
        "path"
      ],
      "title": "read_fileArguments"
    }
  }
}


## 2. 从模型意图路由到真实动作

分派器先区分本地 Skill 工具与已发现的 MCP 工具。未声明名字不能因为模型给出了它，就被执行。工具错误也需要成为模型下一步可见的观察。

下面代码直接展示实际实现，之后的循环会调用这个函数。


In [4]:
async def dispatch(client, library, name, args, remote_names):
    if not isinstance(args, dict): raise ValueError('参数应为 JSON 对象')
    if name == 'activate_skill' and set(args) == {'name'}:
        return {'ok': True, 'instructions': library.activate(args['name'])}
    if name == 'read_skill_resource' and set(args) == {'name','path'}:
        return {'ok': True, 'text': library.read_resource(args['name'], args['path'])}
    if name.startswith('mcp_') and name[4:] in remote_names:
        result = await client.call_tool(name[4:], args)
        return {'ok': not result.is_error,
                'mcp_result': result.model_dump(mode='json', by_alias=True, exclude_none=True)}
    raise ValueError('工具未声明或参数字段不匹配')


## 3. 回到熟悉的循环

阅读时先找四个位置：构建初始消息、发现工具、请求模型、执行并追加观察。Skill 目录开始只提供元数据；正文要等 `activate_skill` 才进入后续请求。

预算、重复调用 ID、模型错误和最终文本都各有停止原因。最终文本不等于成功证明。这里仍是教学最小实现，第六节再加入可持续任务状态和恢复。


In [5]:
async def run_skill_agent(goal, model, max_turns=7, max_tools=12):
    if max_turns < 1 or max_tools < 1: raise ValueError('预算必须为正')
    library = SkillLibrary(ROOT / '05 MCP 与 Skills 能力扩展/代码/skills')
    messages = [{'role': 'system', 'content': SYSTEM + '\n可用 Skill 元数据：' +
                 json.dumps(library.catalog(), ensure_ascii=False)}, {'role': 'user', 'content': goal}]
    trace, wire, seen = [], [], set()
    executed = 0
    def finish(status, answer=''):
        return {'status': status, 'answer': answer, 'tool_calls': executed,
                'messages': messages, 'trace': trace, 'skill_events': library.events, 'mcp_events': wire}
    async with connect(wire) as client:
        discovered = await client.list_tools()
        remote_names = {t.name for t in discovered.tools}
        declarations = SKILL_TOOLS + mcp_declarations(discovered.tools)
        for turn in range(1, max_turns + 1):
            try: response = await asyncio.to_thread(model, messages, declarations)
            except ModelError: return finish('model_error')
            messages.append(response)
            calls = response.get('tool_calls') or []
            if not calls:
                return finish('final_answer' if response.get('content','').strip() else 'empty_answer',
                              response.get('content') or '')
            # 先核对整组 ID，避免执行一半后才发现重复请求。
            ids = [c.get('id') for c in calls]
            if any(not isinstance(i,str) or not i for i in ids) or len(set(ids)) != len(ids) or seen.intersection(ids):
                return finish('invalid_call_ids')
            seen.update(ids)
            if executed + len(calls) > max_tools: return finish('tool_budget')
            for call in calls:
                function = call.get('function', {})
                name = function.get('name', '')
                try:
                    args = json.loads(function.get('arguments',''))
                    result = await dispatch(client, library, name, args, remote_names)
                except (ValueError, TypeError, KeyError, PermissionError, OSError, MCPError) as exc:
                    result = {'ok': False, 'error_type': type(exc).__name__,
                              'message': '调用未完成，请检查工具、参数或可读范围'}
                executed += 1
                trace.append({'turn': turn, 'call_id': call['id'], 'name': name, 'result': result})
                messages.append({'role': 'tool', 'tool_call_id': call['id'],
                                 'content': json.dumps(result, ensure_ascii=False)})
        return finish('turn_budget')


## 4. 先隔离连接与路由问题

固定动作序列依次激活 Skill、读取参考文件、通过 MCP 读源码。它会真的执行 MCP 服务和本地文件读取，但动作顺序是课程为隔离连接与路由问题而预设的，不能说明模型会自主作出这些选择。


In [6]:
replay = ReplayModel([
    tool_response("s1", "activate_skill", {"name": "repo-guide"}),
    tool_response("s2", "read_skill_resource", {"name": "repo-guide", "path": "references/conditions.md"}),
    tool_response("m1", "mcp_read_file", {"path": "config.py"}),
    {"role": "assistant", "content": "路由回放结束，答案正确性另行检查。"},
])
replayed = await run_skill_agent("验证路由", replay)
print(replayed["status"], replayed["skill_events"])
print([(x["name"],x["result"]["ok"]) for x in replayed["trace"]])
assert replayed["tool_calls"] == 3 and all(x["result"]["ok"] for x in replayed["trace"])


final_answer [{'stage': 'discover', 'names': ['repo-guide']}, {'stage': 'activate', 'name': 'repo-guide'}, {'stage': 'resource', 'name': 'repo-guide', 'path': 'references/conditions.md'}]
[('activate_skill', True), ('read_skill_resource', True), ('mcp_read_file', True)]


### 同一条循环中，本地 Skill 与远端工具各走哪条路

配套课件：L05.07-S01、L05.07-S02。

回放已经真实执行两次 Skill 加载和一次 MCP 读取。我们把逻辑工具轨迹与实际协议调用并列：模型的一次工具请求可能由本地加载器处理，也可能经 MCP 客户端发往服务进程。

In [7]:
print("逻辑工具轨迹:")
for case_step in replayed["trace"]:
    case_name = case_step["name"]
    print(case_name, "→", "MCP 服务" if case_name.startswith("mcp_") else "本地 Skill 加载器")
case_rpc_calls = [e["message"] for e in replayed["mcp_events"]
                  if e["direction"] == "client_to_server"
                  and e.get("message", {}).get("method") == "tools/call"]
print("真正发出的 tools/call:", len(case_rpc_calls))
assert len(replayed["trace"]) == 3 and len(case_rpc_calls) == 1


逻辑工具轨迹:
activate_skill → 本地 Skill 加载器
read_skill_resource → 本地 Skill 加载器
mcp_read_file → MCP 服务
真正发出的 tools/call: 1


**结果解读**

逻辑轨迹有 3 次工具动作，MCP tools/call 只有 1 次；差额来自本地 Skill 加载。发现工具的 tools/list 另计，不能把全部协议消息都算成实际读取。

**小练习**

若 read_skill_resource 失败，应先检查 SkillLibrary 还是远端 read_file？从哪个记录开始定位？

<details><summary>完成后展开参考分析</summary>

先按逻辑工具名定位本地加载路径和返回错误；不要因为它处于联合 Agent 中就归因到 MCP 连接。

</details>

公式与实现来源见 [配套素材来源](../../资料来源.md)。

## 5. 看真实模型怎样使用这些能力

默认读取本次课前预跑保存的一次真实 DeepSeek 联调。先看加载事件和工具轨迹，再看最终回答。记录中的模型名只表示本次运行条件。模型可能激活 Skill，也可能漏用；要从记录判断，不能根据系统提示就声称实际发生。

若要新请求，先本地配置 `DEEPSEEK_API_KEY`，再主动开启下面开关。程序会打印本次选择和当前账号可用的模型，并在旧名称已下线时要求先更新 `DEEPSEEK_MODEL`。真实循环至多 7 次模型调用、12 次工具调用，每次最多 1000 输出 Token，无自动重试；每次结果可能不同。


In [8]:
RUN_LIVE = False
record_path = CHAPTER / "数据/实验记录/deepseek-mcp-skills.json"
saved = json.loads(record_path.read_text(encoding='utf-8'))
selected_model = os.environ.get("DEEPSEEK_MODEL", "deepseek-flash")
print("本次选择的模型:", selected_model)
if RUN_LIVE:
    available_models = list_deepseek_models()
    print("当前账号可用模型:", available_models)
    if selected_model not in available_models:
        raise ModelError("所选模型不在当前账号的可用列表中；请先更新 DEEPSEEK_MODEL")
    model = DeepSeekModel(selected_model)
    current = await run_skill_agent(saved["goal"], model)
    usage = model.usage
else:
    current, usage = saved["result"], saved["usage"]
print("本次记录使用的模型:", selected_model if RUN_LIVE else saved["model"])
print("本次数据:", "新真实请求" if RUN_LIVE else "本次课前真实请求记录")
print("停止原因:", current["status"])
print("Skill 事件:", current["skill_events"])
print("实际工具:", [x["name"] for x in current["trace"]])
print(current["answer"])
print("总 Token:", sum(x.get("total_tokens",0) for x in usage))


本次选择的模型: deepseek-flash
本次记录使用的模型: deepseek-flash
本次数据: 本次课前真实请求记录
停止原因: final_answer
Skill 事件: [{'stage': 'discover', 'names': ['repo-guide']}, {'stage': 'activate', 'name': 'repo-guide'}, {'stage': 'resource', 'name': 'repo-guide', 'path': 'references/conditions.md'}]
实际工具: ['activate_skill', 'mcp_list_files', 'read_skill_resource', 'mcp_read_file', 'mcp_read_file', 'mcp_read_file']
# 从 main 到配置读取

**调用关系（一条链）**

`main.py:11` → `main.py:5 main()` → `main.py:6 load_settings()` → `config.py:7 load_settings` → 路径构造 `config.py:8` → 存在性判断 `config.py:9` →（不存在则）默认值 `config.py:10` /（存在则）读取 JSON `config.py:11`

---

## 1. 入口怎么走到配置

- `main.py:10-11`：只有在 `__name__ == "__main__"` 时（也就是直接运行本文件时）才调用 `main()`。被当作模块 import 时这条分支不触发。
- `main.py:6`：`main()` 里第一件事就是 `settings = load_settings()`，没有传入任何参数。
- `main.py:7`：拿到的 `settings` 被当字典用，取 `settings['course']` 并打印。这一点决定了后面默认值里为什么必须有一个 `course` 键。

## 2. 配置路径是怎么确定的

- `config.py:8`：
  ```python
  path = Path(os.environ.get("STUDYBOX_CONFIG", "settings

## 6. 核验时分四层

1. 是否发生了真实 MCP 请求并返回结果？
2. 是否确实激活了 Skill、按需读了参考？
3. 文件范围与预算是否由代码遵守？
4. 最终结论是否被读到的源码支持，讲解是否符合用户要求？

前面三层通过，第四层仍可能出错。第二节出现的相对路径和字段过滤错误，正好可用来检验这次回答。不能据一次结果推断 Skill 普遍提升准确率。


In [9]:
# 这里只列出待核验的源码，不用字符串包含关系冒充语义评分。
repo_tools = __import__("agent_lab").RepoTools(ROOT / "02 从大模型到 Agent 组成与最小循环/数据/demo-repo")
for path in ["main.py", "config.py"]:
    data = repo_tools.read_file(path)
    for line in data["lines"]:
        print(f"{path}:{line['line']} {line['text']}")
print("人工核对：调用关系；相对路径依据；文件缺失分支；未执行的行为是否被说成已执行。")


main.py:1 """StudyBox 运行入口。"""
main.py:2 from config import load_settings
main.py:3 
main.py:4 
main.py:5 def main():
main.py:6     settings = load_settings()
main.py:7     print(f"欢迎，{settings['course']}！")
main.py:8 
main.py:9 
main.py:10 if __name__ == "__main__":
main.py:11     main()
config.py:1 """配置读取；本仓库只作为 Agent 阅读资料。"""
config.py:2 import json
config.py:3 import os
config.py:4 from pathlib import Path
config.py:5 
config.py:6 
config.py:7 def load_settings():
config.py:8     path = Path(os.environ.get("STUDYBOX_CONFIG", "settings.json"))
config.py:9     if not path.exists():
config.py:10         return {"course": "大模型应用开发", "timeout": 30}
config.py:11     return json.loads(path.read_text(encoding="utf-8"))
人工核对：调用关系；相对路径依据；文件缺失分支；未执行的行为是否被说成已执行。


## 练习：已声明与未声明的动作

把回放里的工具名改成 `write_file`，检查有无 MCP 写入请求，再把模型预算限制为一轮。记录停止原因，解释“模型提出动作”与“应用执行动作”的区别。


In [10]:
bad = ReplayModel([
    tool_response("bad1", "write_file", {"path":"config.py"}),
    {"role":"assistant", "content":"停止"}])
rejected = await run_skill_agent("边界练习", bad, max_turns=1)
print(rejected["status"], rejected["trace"])
print("实际 MCP 方法:", [e["message"].get("method") for e in rejected["mcp_events"]
    if e["direction"] == "client_to_server"])


turn_budget [{'turn': 1, 'call_id': 'bad1', 'name': 'write_file', 'result': {'ok': False, 'error_type': 'ValueError', 'message': '调用未完成，请检查工具、参数或可读范围'}}]
实际 MCP 方法: ['tools/list']


In [11]:
assert rejected["status"] == "turn_budget"
assert not rejected["trace"][0]["result"]["ok"]
assert not any(e["message"].get("method") == "tools/call" for e in rejected["mcp_events"])
print("模型提出的未注册写工具被拒绝；没有执行 MCP 写入或读取调用。")


模型提出的未注册写工具被拒绝；没有执行 MCP 写入或读取调用。


## 7. 交付：联合运行证据报告

最后把一次运行的模型名、Skill 加载轨迹、MCP 协议请求、工具结果、停止原因与人工核验项收在一起。报告要能区分：本次真实模型记录、本次真实本地工具回放，以及未通过的边界练习。

模型名只是当时条件。未来新跑前仍要先查询账号当前可用模型，再显式选择。


In [12]:
RUN_DIR = ROOT / ".local/student-runs/l05"
RUN_DIR.mkdir(parents=True, exist_ok=True)
model_name_at_run = selected_model if RUN_LIVE else saved["model"]
mcp_methods = [e["message"].get("method") for e in current["mcp_events"]
               if e.get("direction") == "client_to_server" and "message" in e]
joint_report = {
    "artifact": "L05-joint-agent-run-report",
    "run_source": "new_live_request" if RUN_LIVE else "saved_real_deepseek_record",
    "model_name_at_run": model_name_at_run,
    "future_rerun_rule": "list_current_account_models_then_set_DEEPSEEK_MODEL_explicitly",
    "protocol_version": "2026-07-28",
    "skill_package": "repo-guide",
    "stop_status": current["status"],
    "skill_events": current["skill_events"],
    "tool_trace": current["trace"],
    "mcp_methods": mcp_methods,
    "local_replay": {
        "status": replayed["status"],
        "tool_names": [x["name"] for x in replayed["trace"]],
        "all_tools_succeeded": all(x["result"]["ok"] for x in replayed["trace"]),
    },
    "denied_action_exercise": {
        "status": rejected["status"],
        "proposed_tool": rejected["trace"][0]["name"],
        "executed_mcp_tool_call": any(e["message"].get("method") == "tools/call"
                                      for e in rejected["mcp_events"] if "message" in e),
    },
    "human_review_required": [
        "answer_cites_actual_paths_and_lines",
        "relative_path_claim_matches_source",
        "missing_file_branch_is_not_invented",
        "unexecuted_behavior_is_not_reported_as_executed",
    ],
}
joint_json = RUN_DIR / "joint-run-report.json"
joint_json.write_text(json.dumps(joint_report, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
joint_md = RUN_DIR / "joint-run-report.md"
joint_md.write_text(
    "# L05 联合运行证据报告\n\n"
    f"- 记录来源：{joint_report['run_source']}\n"
    f"- 当时模型：{model_name_at_run}\n"
    f"- 停止原因：{current['status']}\n"
    f"- Skill 事件：{[x['stage'] for x in current['skill_events']]}\n"
    f"- 工具轨迹：{[x['name'] for x in current['trace']]}\n"
    "- 结论：MCP 协议、工具和 Skill 加载轨迹可复查；最终答案仍需按源码人工验收。\n",
    encoding="utf-8",
)
print("已生成联合运行报告:", joint_json.relative_to(ROOT), joint_md.relative_to(ROOT))
print("当时模型:", model_name_at_run, "停止原因:", current["status"])


已生成联合运行报告: .local\student-runs\l05\joint-run-report.json .local\student-runs\l05\joint-run-report.md
当时模型: deepseek-flash 停止原因: final_answer


## 本节总结

MCP 让应用发现并调用跨进程工具；Agent Skill 组织可复用的任务指引和按需参考；宿主把二者送入 Agent 循环，并实际约束执行范围。Skill 和工具都可用，仍要核验最终事实。

本节生成 `joint-run-report.json` 和 Markdown 摘要，保留当时模型名、工具与 Skill 轨迹、边界练习和人工验收项。

下一节以 Harness 视角汇总：任务如何保存、怎样中断恢复、资源如何受控、出了问题从哪里找证据。

来源与版本详见 [课程单元 README](../README.md)。
